In [7]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import curve_fit
from pathlib import Path
from sklearn.metrics import r2_score

# Define the Weibull function for fitting vulnerability curves
def weibull_function(x, d, b):
    """
    Weibull function for vulnerability curves: % Loss = 100 * (1 - exp(-((-x/b)^d)))
    x: PSY (MPa, negative values)
    d: Shape parameter
    b: Scale parameter (positive value related to the scale of PSY)
    Returns: % Loss
    """
    return 100*(1-np.exp(-((-x/d)**b))) #100 * (1 - 

# Function to read and plot vulnerability curves with best-fit lines
def plot_vulnerability_curves(file_path, title):
    # Read the CSV file
    df = pd.read_csv(file_path)
    
    # Get unique curve types (c1, c2, etc.)
    curves = df['Curve'].unique()
    
    # Create a Plotly figure
    fig = go.Figure()
    
    # List to store results for the table
    results = []
    
    # Iterate over each curve type
    for curve in curves:
        # Filter data for the current curve
        df_curve = df[df['Curve'] == curve]
        
        # Sort the data by 'PSY (MPa)' in ascending order (most negative to least negative)
        df_curve = df_curve.sort_values(by='PSY (Mpa)', ascending=True)
        
        # Get the material for this curve (assuming it's consistent within a curve)
        material = df_curve['Material'].iloc[0]  # Take the first occurrence
        type = df_curve['Type'].iloc[0]  # Take the first occurrence

        # Create the label as "curve-material"
        label = f"{curve}-{material}"
        
        # Extract x (PSY) and y (% Loss) for fitting
        x_data = df_curve['PSY (Mpa)']
        y_data = df_curve['% Loss']
        
        # # Inside the for loop, before fitting
        # print(f"Processing curve: {curve}, Material: {material}, Shape of df_curve: {df_curve.shape}")
        
        # Fitting block
        try:
            # Remove NaN and inf values
            mask = ~np.isnan(x_data) & ~np.isnan(y_data) & np.isfinite(x_data) & np.isfinite(y_data)
            x_data_clean = x_data[mask]
            y_data_clean = y_data[mask]
            
            if len(x_data_clean) < 2:
                raise RuntimeError("Insufficient valid data points for fitting")
            
            # Use raw x_data (negative values) directly since the function handles -x
            # Initial guess based on your observations (d~5, b~5 for wide Douglas Fir)
            initial_guess = [5, 5]  # d=5, b=5 as a starting point
            bounds = ([0.1, 0.1], [10, 10])  # Constrain d and b to positive values
            
            popt, _ = curve_fit(weibull_function, x_data_clean, y_data_clean, 
                                p0=initial_guess, bounds=bounds, maxfev=10000)
            d, b = popt
            
            # Calculate R²
            y_pred = weibull_function(x_data_clean, d, b)
            r2 = r2_score(y_data_clean, y_pred)
            
            # Generate smooth x values for the fitted curve
            x_smooth = np.linspace(min(x_data_clean), max(x_data_clean), 100)
            y_smooth = weibull_function(x_smooth, d, b)
            
            # Find x-value where % Loss is closest to 99.9%
            x_extended = np.linspace(min(x_data_clean)*2, max(x_data_clean) * 2, 1000)
            y_extended = weibull_function(x_extended, d, b)
            idx_100 = np.argmin(np.abs(y_extended - 99)) #change -99 to whatever threshold close to 100 
            x_at_100 = x_extended[idx_100]
            percent_at_100 = y_extended[idx_100]
            
            # Store results
            results.append({
                'Curve': curve,
                'Type': type,
                'd (Shape)': round(d, 3),
                'b (Scale)': round(b, 3),
                'R²': round(r2, 3),
                'PSY at 100% Loss (MPa)': round(x_at_100, 3),
                '% Loss at 100% PSY': round(percent_at_100, 3)
            })
            
            # Add the best-fit curve
            fig.add_trace(
                go.Scatter(
                    x=x_smooth,
                    y=y_smooth,
                    mode='lines',
                    name=f"{label} (fit)",
                    line=dict(dash='dash'),
                    hovertemplate='<b>%{y:.1f}</b> % Loss at <b>%{x:.2f}</b> MPa<br>Curve: %{text}',
                    text=[label] * len(x_smooth),
                )
            )
        except RuntimeError as e:
            print(f"Could not fit curve for {label}: {e}")
            results.append({
                'Curve-Material': label,
                'd (Shape)': None,
                'b (Scale)': None,
                'R²': None,
                'PSY at ~100% Loss (MPa)': None
            })
        
        # Add scatter trace for the original data points
        fig.add_trace(
            go.Scatter(
                x=df_curve['PSY (Mpa)'],
                y=df_curve['% Loss'],
                mode='markers',
                name=f"{label} (data)",
                hovertemplate='<b>%{y}</b> % Loss at <b>%{x}</b> Mpa<br>Curve: %{text}',
                text=[label] * len(df_curve),
            )
        )
    
    # Update layout with titles and labels
    fig.update_layout(
        xaxis_title="Water Potential (PSY, Mpa)",
        yaxis_title="% Loss of Conductivity",
        legend_title="Curve#-Material(type)",
        showlegend=True
    )
    
    # Display the plot
    fig.show()
    
    # Create and print results table
    results_df = pd.DataFrame(results)
    print("\nFit Parameters and Metrics:")
    print(results_df.to_string(index=False))
    
    return fig

# File paths for the CSV files
from pathlib import Path
path_cwd = Path.cwd()
path_input = str(path_cwd) + '/Input_files/Literature_VC/'
df_file = path_input + "DF_VC.csv"  # Douglas Fir data
es_file = path_input + "ES_VC.csv"  # Engelmann Spruce data

# Plot for Douglas Fir
fig_es=plot_vulnerability_curves(df_file, "Vulnerability Curves for Douglas Fir")

# Plot for Engelmann Spruce
fig_df=plot_vulnerability_curves(es_file, "Vulnerability Curves for Engelmann Spruce")


# Paths to save figures
path_output_graphs = str(path_cwd) + '/Output_graphs/'
path_graphs=path_output_graphs+'Graphs/'
path_vc=path_graphs+'Literature_VC/'


# Save PNG
fig_es.write_image(path_vc+'VC_DF.png',scale=8)
fig_df.write_image(path_vc+'VC_ES.png',scale=8)

# Save HTML
fig_es.write_html(path_vc + 'VC_DF.html', include_plotlyjs='cdn')
fig_df.write_html(path_vc + 'VC_ES.html', include_plotlyjs='cdn')


Fit Parameters and Metrics:
Curve   Type  d (Shape)  b (Scale)    R²  PSY at 100% Loss (MPa)  % Loss at 100% PSY
   c1 branch      4.584      2.676 0.831                  -8.112              98.999
   c2 branch      3.942     10.000 0.793                  -4.591              98.981
   c3 branch      5.746      6.328 0.997                  -7.314              98.997
   c4 branch      4.906      4.073 0.976                  -7.144              99.014
   c5   root      4.181      7.147 0.996                  -5.178              99.004
   c6   root      2.393      1.355 0.983                  -7.384              98.997
   c7   stem      5.060      5.448 0.979                  -6.698              99.003
   c8   root      2.810      2.829 0.984                  -4.819              98.993
   c9   stem      5.821      3.725 0.957                  -8.773              99.003
  c10   root      4.544      2.923 0.938                  -7.661              99.000
  c11   stem      5.866      3.968 0


Fit Parameters and Metrics:
Curve Type  d (Shape)  b (Scale)    R²  PSY at 100% Loss (MPa)  % Loss at 100% PSY
   c1 stem      4.558      4.479 0.909                  -6.408              98.993
   c2 root      1.932      2.308 0.982                  -3.748              99.012
   c3 root      1.499      1.002 0.889                  -6.883              99.000
   c4 root      3.278      1.924 0.920                  -7.245              98.993
   c5 root      0.942      0.956 0.976                  -4.656              99.002
